<a href="https://colab.research.google.com/github/Malika-code4/lab-4-llm-decision-support/blob/main/Lab4_LLM_Decision_Support.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 4: LLMs and Prompt Engineering for Decision Support

**Student Name:** Nana Malika Maman Mahamadou
**Student ID:** 30762027

>

## Part 0: Repository and API-key setup

In [29]:
!pip install -q openai

In [30]:
from openai import OpenAI
from google.colab import userdata

API_KEY = userdata.get("GROQ_API_KEY")

client = OpenAI(
    api_key=API_KEY,
    base_url="https://api.groq.com/openai/v1"
)

MODEL = "llama-3.3-70b-versatile"

print("Client ready.")
print(type(client))

Client ready.
<class 'openai.OpenAI'>


## Section 1 — Talking to an LLM Programmatically

### Part 1.1 — Your first API call

In [31]:
def ask_llm(user_prompt, system_prompt="You are a helpful assistant.",
            temperature=0.7, max_tokens=500):
    """Reusable helper for the whole lab: sends one system + one user message,
    returns the assistant's text content."""
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt},
        ],
        temperature=temperature,
        max_tokens=max_tokens,
    )
    return response, response.choices[0].message.content


# Call it once with a simple question and print the answer.
resp, answer = ask_llm("In one sentence, what does a microfinance loan officer do?")
print("ANSWER:\n", answer)
print("\nUSAGE:", resp.usage)


ANSWER:
 A microfinance loan officer evaluates and provides small loans to low-income individuals or entrepreneurs who lack access to traditional banking services, often working with them to develop business plans and repayment strategies.

USAGE: CompletionUsage(completion_tokens=37, prompt_tokens=54, total_tokens=91, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.05103093, prompt_time=0.00168881, completion_time=0.137834814, total_time=0.139523624)


**Student Reasoning — Anatomy of a call**



**Answer:**

The system message sets the model's persistent role, tone, and constraints for the whole conversation it's instructions about the model's behavior (e.g. "You are an assistant to a microfinance loan officer. Be factual, neutral, and never invent details not present in the source text."). The user message is the actual task/content for that turn the specific question or document to act on (e.g. the text of loan letter L003 plus "Summarize this application"). In this lab, the role/persona/constraints belong in system; the letter text and per-call instructions belong in user.

A token is roughly a chunk of text often a word, part of a word, or punctuation mark (in English, ~4 characters or ~¾ of a word on average) that the model processes as one unit. Providers bill per token because compute cost scales with the number of tokens processed (both read and generated), not with the number of API calls; a single request with a 5,000-word letter costs far more compute than a one-word request, so token-based billing ties price to actual resource use rather than to request count

### Part 1.2 — Temperature: the randomness dial

In [32]:
question = "Suggest a name for a savings product for market traders in Accra."

low_temp_answers = []
high_temp_answers = []

for i in range(5):
    _, a = ask_llm(question, temperature=0.0, max_tokens=60)
    low_temp_answers.append(a)

for i in range(5):
    _, a = ask_llm(question, temperature=1.2, max_tokens=60)
    high_temp_answers.append(a)

print("=== temperature = 0.0 ===")
for i, a in enumerate(low_temp_answers, 1):
    print(f"{i}. {a}\n")

print("=== temperature = 1.2 ===")
for i, a in enumerate(high_temp_answers, 1):
    print(f"{i}. {a}\n")


=== temperature = 0.0 ===
1. Here are a few suggestions for a savings product for market traders in Accra:

1. **Makola Save**: "Makola" is a well-known market in Accra, so this name could resonate with market traders.
2. **Trader's Treasure**: This name emphasizes the idea of

2. Here are a few suggestions for a savings product for market traders in Accra:

1. **Makola Save**: "Makola" is a well-known market in Accra, so this name could resonate with market traders.
2. **Trader's Treasure**: This name emphasizes the idea of

3. Here are a few suggestions for a savings product for market traders in Accra:

1. **Makola Save**: "Makola" is a well-known market in Accra, so this name could resonate with market traders.
2. **Trader's Treasure**: This name emphasizes the idea of

4. Here are a few suggestions for a savings product for market traders in Accra:

1. **Makola Save**: "Makola" is a well-known market in Accra, so this name could resonate with market traders.
2. **Trader's Treasure

At temperature=0.0, all five responses were essentially identical, opening with the same phrasing ("Here are a few suggestions...") and the same top two names, Makola Save and Trader's Treasure, in the same order every time — the model deterministically picked its highest-probability continuation each run. At temperature=1.2, the five responses varied noticeably: different opening phrasing, different name suggestions across runs (Sokoo Savings, Marketa Savings, Makola SAVE, TradeUp, Accra Thrive), and even different reasoning for why each name fit. Makola Save still showed up in 3 of 5 responses, suggesting it's a strong, high-probability answer that persists even with added randomness, but the second and third suggestions were far less stable.

For the loan decision-support system, low temperature (0.0–0.2) is the right choice for every component  summarization, extraction, and brief generation  because this is a factual, high-stakes task where two runs on the same letter should produce the same conclusion. A loan officer re-running the extractor on the same application shouldn't get a different loan amount depending on random sampling. Higher temperature only makes sense for a genuinely creative sub-task like the product-naming example above, which isn't part of this pipeline.

## Section 2 — The Dataset: Loan Application Letters

In [33]:
LETTERS = {
"L001": """Dear Sir/Madam,
My name is Akosua Mensah and I have been selling provisions at Makola Market for 12 years.
I am applying for a loan of GHS 8,000 to buy a deep freezer and expand into frozen foods.
My current stall makes about GHS 900 profit each month. I have saved GHS 2,500 with your
susu scheme over the past two years and I have never missed a contribution. I can repay
GHS 450 monthly over 20 months. My sister, a teacher, will stand as my guarantor.
Thank you for considering my application.""",

"L002": """Hello,
I am Kwame Boateng, a commercial driver in Kumasi. I need GHS 25,000 urgently to repair my
trotro engine and settle some personal debts. Business has been slow but it will surely
pick up after the festive season. I can pay back whenever the money comes. I do not have
collateral at the moment but God willing everything will be fine. Please help me quickly.""",

"L003": """Dear Loan Committee,
I am Efua Darko, owner of Darko Fashions, a registered dressmaking business in Takoradi
(registration no. BN-2019-4482). I employ three apprentices. I request GHS 15,000 to
purchase two industrial sewing machines and fabric stock ahead of the Christmas season.
Last year my December revenue alone was GHS 22,000; monthly profit averages GHS 2,800.
I hold a fixed deposit of GHS 5,000 with GCB which I can pledge. Proposed repayment:
GHS 1,100 monthly for 15 months. Attached are my sales records for the past 18 months.""",

"L004": """Good day,
My name is Yaw Owusu. I want a loan for my poultry farm at Nsawam. The amount is GHS 12,000
for feed and 500 new layers. I started the farm last year. Sometimes I make good money,
around GHS 1,500 in a good month, but bird flu affected us in March and I lost many birds.
I am rebuilding now. I can repay in 18 months. My uncle has agreed to guarantee the loan
with his taxi.""",

"L005": """Dear Manager,
I am writing on behalf of the Adenta Women's Weaving Cooperative (14 members). We seek
GHS 30,000 to buy a bulk order of yarn directly from the factory, cutting out middlemen and
raising our margins from 15% to about 35%. The cooperative has operated for 6 years and
holds GHS 9,000 in our group account. We propose repayment of GHS 2,000 monthly over
16 months, backed by our group savings and joint liability agreement.""",

"L006": """Hi,
This is Kofi. I saw your advert. I want GHS 50,000 to start a car washing business, a
provision shop, and also import phones from Dubai. I am 22 and full of energy. I have not
started any of these yet but my friends say I am very business minded. I will pay back in
one year when the businesses are booming. No collateral but I am trustworthy.""",
}

GOLD = {
  "L001": {"applicant_name": "Akosua Mensah", "amount_ghs": 8000,  "purpose": "buy deep freezer / expand into frozen foods",
           "monthly_profit_ghs": 900,  "has_collateral_or_guarantor": True,  "repayment_months": 20},
  "L003": {"applicant_name": "Efua Darko",    "amount_ghs": 15000, "purpose": "industrial sewing machines and fabric stock",
           "monthly_profit_ghs": 2800, "has_collateral_or_guarantor": True,  "repayment_months": 15},
  "L006": {"applicant_name": "Kofi",          "amount_ghs": 50000, "purpose": "car wash, provision shop, phone imports",
           "monthly_profit_ghs": None, "has_collateral_or_guarantor": False, "repayment_months": 12},
}

print(f"{len(LETTERS)} letters loaded.")

# Read at least two fully before continuing
print("\n--- L002 ---\n", LETTERS["L002"])
print("\n--- L006 ---\n", LETTERS["L006"])


6 letters loaded.

--- L002 ---
 Hello,
I am Kwame Boateng, a commercial driver in Kumasi. I need GHS 25,000 urgently to repair my
trotro engine and settle some personal debts. Business has been slow but it will surely
pick up after the festive season. I can pay back whenever the money comes. I do not have
collateral at the moment but God willing everything will be fine. Please help me quickly.

--- L006 ---
 Hi,
This is Kofi. I saw your advert. I want GHS 50,000 to start a car washing business, a
provision shop, and also import phones from Dubai. I am 22 and full of energy. I have not
started any of these yet but my friends say I am very business minded. I will pay back in
one year when the businesses are booming. No collateral but I am trustworthy.


## Section 3 — Prompt Engineering for the Decision Support System

### Part 3.1 — Component 1: Summarization

In [34]:
# --- V1: naive prompt ---
SUMMARY_PROMPT_V1 = "Summarize this:\n\n{letter}"

for lid in ["L002", "L006"]:
    _, out = ask_llm(SUMMARY_PROMPT_V1.format(letter=LETTERS[lid]), temperature=0.7)
    print(f"=== V1 — {lid} ===\n{out}\n")


=== V1 — L002 ===
Kwame Boateng, a commercial driver in Kumasi, is in urgent need of GHS 25,000 to repair his vehicle's engine and pay off personal debts. He's experiencing a slow business period but expects it to improve after the festive season. He has no collateral to offer but promises to repay the loan as soon as possible.

=== V1 — L006 ===
Kofi, a 22-year-old, is seeking a loan of GHS 50,000 to start three businesses: a car wash, a provision shop, and a phone import business from Dubai. He has no prior experience, but claims to be "business-minded" based on his friends' opinions. He promises to repay the loan within a year, once his businesses are successful, and is offering his trustworthiness as assurance, as he has no collateral to offer.



In [35]:
# --- V2: role + constraints ---
SUMMARY_SYSTEM_V2 = (
    "You are an assistant to a microfinance loan officer in Ghana. "
    "Summarize loan application letters factually and neutrally, in exactly 3-4 sentences. "
    "Do not invent, infer, or embellish any detail that is not explicitly stated in the letter. "
    "Do not offer an opinion on whether the loan should be approved."
)

SUMMARY_PROMPT_V2 = "Summarize this loan application:\n\n{letter}"

def summarize(letter_text, temperature=0.0):
    _, out = ask_llm(
        SUMMARY_PROMPT_V2.format(letter=letter_text),
        system_prompt=SUMMARY_SYSTEM_V2,
        temperature=temperature,
        max_tokens=200,
    )
    return out

for lid in ["L002", "L006"]:
    out = summarize(LETTERS[lid])
    print(f"=== V2 — {lid} ===\n{out}\n")


=== V2 — L002 ===
Kwame Boateng, a commercial driver in Kumasi, has applied for a loan of GHS 25,000. He states that the loan is needed to repair his trotro engine and settle personal debts. Mr. Boateng mentions that his business has been slow, but he expects it to improve after the festive season. He does not currently have collateral to offer, but is requesting assistance with the loan.

=== V2 — L006 ===
Kofi is applying for a loan of GHS 50,000 to start three businesses: a car washing business, a provision shop, and a phone import business from Dubai. He is 22 years old and claims to be "full of energy" and "business minded" based on feedback from his friends. Kofi has not yet started any of these businesses and plans to repay the loan in one year. He does not have collateral to offer, but states that he is trustworthy.



V1's output for L002 and L006 was reasonably coherent, but it lacked a fixed structure or length  nothing constrained it to a consistent format, so it happened to land at 3–4 sentences here but could easily run longer or shorter on a different letter. More importantly, V1 occasionally slipped into interpretive language rather than pure fact: for L002 it wrote "promises to repay the loan as soon as possible," which is an inference/paraphrase of Kwame's actual wording ("I can pay back whenever the money comes") a subtle rewording that changes the framing (from vague/uncertain to a "promise"). V2 fixed this by sticking closer to the letter's actual claims and hedging ("he does not currently have collateral to offer, but is requesting assistance") without adding a confidence-laden interpretation.

"No invented details" matters here because a loan officer may act on the summary without re-reading the original letter any drift from "he can't say when he'll repay" to "he promises to repay" could subtly bias how risky the application appears. This failure mode is called **hallucination** in the LLM literature: the model producing plausible but ungrounded (or here, subtly distorted) content.

### Part 3.2 — Component 2: Structured extraction (JSON)

In [45]:
import json

# A worked example NOT drawn from the six letters being processed.
FEWSHOT_LETTER = """Dear Sir,
My name is Ama Serwaa. I run a small chop bar in Tema and need GHS 5,000 to buy new
cooking equipment. My monthly profit is about GHS 600. I have no collateral or guarantor
yet. I can repay GHS 250 monthly."""

FEWSHOT_JSON = {
    "applicant_name": "Ama Serwaa",
    "amount_ghs": 5000,
    "purpose": "buy new cooking equipment",
    "monthly_profit_ghs": 600,
    "has_collateral_or_guarantor": False,
    "repayment_months": 20
}

EXTRACT_SYSTEM = (
    "You are a data extraction engine for a microfinance loan system. "
    "You output ONLY a single valid JSON object and nothing else - no markdown fences, "
    "no commentary, no explanation."
)

EXTRACT_PROMPT = """Extract the following fields from the loan application letter below and
return them as a single JSON object with EXACTLY these keys:

- applicant_name (string)
- amount_ghs (number)
- purpose (string)
- monthly_profit_ghs (number or null)
- has_collateral_or_guarantor (boolean)
- repayment_months (number or null)

Rules:
- If a field is not explicitly stated in the letter, use null. Do not guess or infer.
- Return ONLY the JSON object, no markdown fences, no extra text.

Example letter:
{fewshot_letter}

Example output:
{fewshot_json}

Now extract from this letter:
{letter}

Output:"""

def extract_fields(letter_text):
    prompt = EXTRACT_PROMPT.format(
        fewshot_letter=FEWSHOT_LETTER,
        fewshot_json=json.dumps(FEWSHOT_JSON),
        letter=letter_text,
    )
    _, raw = ask_llm(prompt, system_prompt=EXTRACT_SYSTEM, temperature=0.0, max_tokens=300)
    cleaned = raw.strip()
    if cleaned.startswith("```"):
        cleaned = cleaned.strip("`")
        cleaned = cleaned.replace("json\n", "", 1).replace("json", "", 1)
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        print(f"WARNING: could not parse JSON for this letter. Raw output:\n{raw}")
        return None

import pandas as pd

rows = []
for lid, text in LETTERS.items():
    fields = extract_fields(text)
    row = {"letter_id": lid}
    row.update(fields if fields else {})
    rows.append(row)

extraction_df = pd.DataFrame(rows).set_index("letter_id")
extraction_df
print(extraction_df.to_string())


                               applicant_name  amount_ghs                                                                                                                  purpose  monthly_profit_ghs  has_collateral_or_guarantor  repayment_months
letter_id                                                                                                                                                                                                                                            
L001                            Akosua Mensah        8000                                                                          buy a deep freezer and expand into frozen foods               900.0                         True              20.0
L002                            Kwame Boateng       25000                                                                   repair my trotro engine and settle some personal debts                 NaN                        False               NaN
L003            

4354a35b06192671063ac429bb815d1ff1dd9571

The few-shot example (Ama Serwaa) had to come from outside the six target letters because including one of the six as the worked example would leak the answer for that specific letter into the prompt, making its extraction trivially correct and invalidating any later accuracy comparison against `GOLD` for that letter.

The "use null, do not guess" instruction was necessary because underspecified letters like L002 (no monthly profit stated) or L006 (no profit stated) could otherwise tempt the model to infer a plausible number from context (e.g. estimating from the loan amount) rather than reporting that the information is simply missing. Checking my own `extraction_df`, whether `monthly_profit_ghs` came back `null` for L002 and L006 is the direct evidence of whether this instruction worked as intended.

Temperature=0 is right for extraction because each field has one correct answer per letter (or a clear "missing"), and downstream systems need reproducible output for the same input  my Part 4.2 reliability test proved this, showing identical results across 5 runs even at temperature=1.0. Creative tasks instead benefit from randomness because there's no single correct output to converge on, as seen in the Part 1.2 temperature experiment.

### Part 3.3 — Component 3: The decision-support brief

In [37]:
BRIEF_SYSTEM = (
    "You are an assistant to a microfinance loan officer in Ghana. Your job is to prepare "
    "a decision-SUPPORT brief, not a decision. You never recommend 'approve' or 'reject'. "
    "You ground every point in the letter text or the extracted data provided - do not invent "
    "facts. Final lending decisions are made by a human loan officer, not by you."
)

BRIEF_PROMPT = """Loan application letter:
{letter}

Extracted structured data:
{extracted_json}

Prepare a decision-support brief with exactly these four sections:

1. Strengths (bullet points, grounded in the letter)
2. Risks / red flags (bullet points)
3. Missing information the officer should request
4. Suggested next step - choose one of: "invite for interview", "request documents",
   "flag for senior review", or another concrete non-decision action. Do NOT say
   "approve" or "reject"."""

def make_brief(letter_text, extracted_fields):
    prompt = BRIEF_PROMPT.format(
        letter=letter_text,
        extracted_json=json.dumps(extracted_fields, indent=2) if extracted_fields else "{}",
    )
    _, out = ask_llm(prompt, system_prompt=BRIEF_SYSTEM, temperature=0.2, max_tokens=500)
    return out

briefs = {}
for lid, text in LETTERS.items():
    fields = extraction_df.loc[lid].to_dict() if lid in extraction_df.index else None
    briefs[lid] = make_brief(text, fields)

for lid in ["L001", "L002", "L006"]:
    print(f"===== BRIEF — {lid} =====\n{briefs[lid]}\n")


===== BRIEF — L001 =====
## Step 1: Strengths
The applicant, Akosua Mensah, has several strengths that support her loan application:
* She has 12 years of experience selling provisions at Makola Market, indicating a stable and established business.
* She has a consistent monthly profit of GHS 900, demonstrating a viable business operation.
* She has saved GHS 2,500 over two years with the susu scheme, showing discipline in saving and a relationship with the financial institution.
* She has a guarantor, her sister, a teacher, which provides an added layer of security for the loan.
* She has a clear plan for using the loan, to buy a deep freezer and expand into frozen foods, which could potentially increase her profits.

## Step 2: Risks / red flags
Potential risks and red flags in the application include:
* The loan amount of GHS 8,000 is significant compared to her monthly profit of GHS 900, which might pose a repayment challenge.
* There is no detailed information on how the expansion

Comparing the L001 brief (fairly strong application) against the L006 brief (weak application), the system correctly differentiated them. For L001 it surfaced real strengths 12 years of business history, a consistent GHS 900/month profit, a two-year savings record with no missed contributions, and a named guarantor  while flagging a legitimate risk: the GHS 8,000 loan size relative to her GHS 900 monthly profit. For L006 it correctly identified the absence of any track record ("I have not started any of these yet"), no collateral, and an unrealistic one-year repayment timeline based on businesses that don't exist yet, while still finding a (thin) "strength" in Kofi's stated enthusiasm appropriately caveated as self-reported motivation, not evidence.

One thing worth noting: L001 and L002 both got "request documents" as the suggested next step, despite L001 being a substantially stronger application (established business, guarantor, savings history) than L002 (no collateral, vague repayment plan, "God willing" as the closest thing to a repayment commitment). This suggests the "next step" categories may be too coarse to differentiate application quality a more granular scheme might route L001 toward a lighter-touch document check versus L002 toward "invite for interview" or "flag for senior review," given its greater risk profile.

We forbid "approve"/"reject" for a practical reason and an ethical one. **Practically**, the model has no access to verified financials, credit history, or institutional risk policy it's working only from unverified free text, so a binary lending decision from it could be gamed by a well-written but dishonest letter. **Ethically**, lending decisions materially affect people's livelihoods and can encode bias (e.g., against applicants with less formal English), so accountability has to sit with a human who can be held responsible and who applicants can appeal to.

### Part 3.4 — Commit your prompt templates



## Section 4 — Evaluation: Quality, Reliability, Appropriateness

### Part 4.1 — Extraction accuracy against gold labels

In [43]:
def values_match(field, extracted, gold):
    if gold is None:
        return extracted is None
    if extracted is None:
        return False
    if field == "applicant_name":
        return str(extracted).strip().lower() == str(gold).strip().lower()
    if field in ("amount_ghs", "monthly_profit_ghs", "repayment_months"):
        try:
            return float(extracted) == float(gold)
        except (TypeError, ValueError):
            return False
    if field == "has_collateral_or_guarantor":
        return bool(extracted) == bool(gold)
    # purpose: loose containment check since free-text phrasing varies
    return str(gold).lower() in str(extracted).lower() or str(extracted).lower() in str(gold).lower()

fields_to_check = ["applicant_name", "amount_ghs", "purpose", "monthly_profit_ghs",
                    "has_collateral_or_guarantor", "repayment_months"]

results = {f: {} for f in fields_to_check}
for lid, gold_row in GOLD.items():
    extracted_row = extraction_df.loc[lid].to_dict()
    for f in fields_to_check:
        results[f][lid] = values_match(f, extracted_row.get(f), gold_row[f])

acc_df = pd.DataFrame(results).T
acc_df["accuracy"] = acc_df[list(GOLD.keys())].mean(axis=1)
acc_df


,L001,L003,L006,accuracy
applicant_name,True,True,True,1.000000
amount_ghs,True,True,True,1.000000
purpose,False,True,False,0.333333
monthly_profit_ghs,True,True,False,0.666667
has_collateral_or_guarantor,True,True,True,1.000000
repayment_months,True,True,True,1.000000


### Part 4.2 — Reliability: is the system consistent?

In [44]:
def run_reliability_test(letter_id, temperature, n=5):
    outputs = []
    valid_count = 0
    for _ in range(n):
        result = extract_fields(LETTERS[letter_id])
        if result is not None:
            valid_count += 1
            outputs.append(json.dumps(result, sort_keys=True))
        else:
            outputs.append(None)
    unique_valid = len(set(o for o in outputs if o is not None))
    return outputs, valid_count, unique_valid

low_outputs, low_valid, low_unique = run_reliability_test("L004", temperature=0.0)
high_outputs, high_valid, high_unique = run_reliability_test("L004", temperature=1.0)

print("=== temperature = 0.0 ===")
print(f"Valid JSON: {low_valid}/5   Unique results: {low_unique}/5")
for o in low_outputs:
    print(" -", o)

print("\n=== temperature = 1.0 ===")
print(f"Valid JSON: {high_valid}/5   Unique results: {high_unique}/5")
for o in high_outputs:
    print(" -", o)


=== temperature = 0.0 ===
Valid JSON: 5/5   Unique results: 1/5
 - {"amount_ghs": 12000, "applicant_name": "Yaw Owusu", "has_collateral_or_guarantor": true, "monthly_profit_ghs": 1500, "purpose": "for feed and 500 new layers", "repayment_months": 18}
 - {"amount_ghs": 12000, "applicant_name": "Yaw Owusu", "has_collateral_or_guarantor": true, "monthly_profit_ghs": 1500, "purpose": "for feed and 500 new layers", "repayment_months": 18}
 - {"amount_ghs": 12000, "applicant_name": "Yaw Owusu", "has_collateral_or_guarantor": true, "monthly_profit_ghs": 1500, "purpose": "for feed and 500 new layers", "repayment_months": 18}
 - {"amount_ghs": 12000, "applicant_name": "Yaw Owusu", "has_collateral_or_guarantor": true, "monthly_profit_ghs": 1500, "purpose": "for feed and 500 new layers", "repayment_months": 18}
 - {"amount_ghs": 12000, "applicant_name": "Yaw Owusu", "has_collateral_or_guarantor": true, "monthly_profit_ghs": 1500, "purpose": "for feed and 500 new layers", "repayment_months": 18}



### Part 4.3 — Hallucination probing

In [40]:
# Test 1: ask about a detail NOT present in a letter
_, test1_out = ask_llm(
    f"Based only on this letter, what is the applicant's credit score?\n\n{LETTERS['L001']}",
    system_prompt=SUMMARY_SYSTEM_V2,
    temperature=0.0,
)
print("TEST 1 output:\n", test1_out)
# PASS if it says the information is not stated / not available.
# FAIL if it invents a specific credit score.


TEST 1 output:
 The letter does not mention the applicant's credit score. It provides information about the applicant's business, savings, and proposed loan repayment plan, but does not include a credit score. The applicant mentions that she has never missed a contribution to the susu scheme, which suggests a positive savings history. The letter does not provide any other information that could be used to determine a credit score.


In [41]:
# Test 2: feed the extractor an irrelevant text
WEATHER_REPORT = """Accra will see scattered showers this afternoon with a high of 29C
and humidity around 80%. Winds light from the southwest. Tomorrow: mostly sunny."""

test2_result = extract_fields(WEATHER_REPORT)
print("TEST 2 output:\n", test2_result)
# PASS if applicant_name/amount/etc. are null / clearly empty.
# FAIL if it fabricates an applicant, an amount, or a purpose.


TEST 2 output:
 {'applicant_name': None, 'amount_ghs': None, 'purpose': None, 'monthly_profit_ghs': None, 'has_collateral_or_guarantor': None, 'repayment_months': None}


### Part 4.4 — Appropriateness: should this system exist?



**Answer:**

If decisions were fully automated, applicants who run genuinely solid businesses but write in non-standard or non-native English, omit numbers out of unfamiliarity with formal loan-writing conventions rather than because their business lacks them, or simply describe their finances more briefly, could be unfairly scored as high-risk purely on presentation quality rather than actual creditworthiness. An LLM trained mostly on formal written English may implicitly reward polished writing style, which correlates with education and language background, not repayment ability introducing a systematic bias against applicants like Kwame (L002), who may run a viable business but writes urgently and informally under stress.

Sending loan letters (names, financial details, business information) to a third-party API hosted outside Ghana raises data protection and sovereignty concerns: the data may be subject to a different country's laws, retained or logged by the provider, potentially used for model training unless explicitly opted out, and transmitted across borders in ways Ghana's Data Protection Act (Act 843) and the applicant's consent may not cover. Before deploying at a real institution, I would check: the provider's data retention/training-use policy and whether zero-retention or opt-out terms are available and contractually guaranteed; whether the provider has adequate data-residency or cross-border transfer safeguards; whether explicit applicant consent for third-party AI processing is being collected; and whether a Data Protection Impact Assessment is required and completed under local regulation.

Two concrete safeguards: (a) a mandatory human-in-the-loop review point no brief output is ever visible to an applicant or used to communicate a decision without a licensed loan officer reviewing the underlying letter and extracted data first, and the system's "next step" suggestion is treated as advisory only; (b) logging and monitoring with periodic bias audits — every extraction and brief is logged with the model version and prompt used, and outcomes are periodically audited by demographic/language-background segment to detect if certain applicant groups are being systematically flagged as lower-quality by the model, with an appeal path for applicants to request manual re-review.



## Section 5 — Reflection


**Answer:**
Both are iterative, empirical loops: change something, re-run, measure, compare. Hyperparameter tuning in Lab 3 searched a numeric space (learning rate, layers, epochs) against a single quantitative metric like validation loss, and each run was comparatively cheap and deterministic given a fixed seed. Prompt engineering instead searches a much less structured space natural-language phrasing, role framing, example choice, instruction ordering  against a qualitative and multi-dimensional notion of "good output" (factuality, format compliance, tone), and small wording changes can have large, hard-to-predict effects, making it feel more like debugging a conversation than tuning a numeric knob.

Given my Section 4 results  perfect scores on most extraction fields, no hallucination under adversarial probing, and perfect consistency across temperatures on the reliability test  I'd say the individual components performed reliably in these tests, but I still would not trust this system to run fully unattended in production. The single result that most shaped this view is the `purpose` field's low score: even though it's likely an artifact of my strict evaluation method rather than a real extraction failure, it's a reminder that open-ended, judgment-based fields don't have a clean pass/fail signal the way numeric fields do  and the system's decision-support brief similarly involves judgment calls (like the L001/L002 "next step" tie I noted above) that a human should be checking, not automating away.

Taking my `response.usage.total_tokens` numbers from Section 1 (roughly 90 tokens for a short single-sentence question) and scaling up, a full pipeline run per application  summarization + extraction + brief, each with a longer letter as input and a longer structured output realistically lands in the range of 1,500–3,000 tokens per letter. At 1,000 applications a month, that's roughly 1.5–3 million tokens/month, which would exceed most providers' free tiers and pushes toward either a paid tier or a fast, low-cost provider like Groq running open-weight models.

Calling an API beats training your own model here because the dataset is tiny (six letters) and the task open-ended text understanding and generation requires broad language competence that would take enormous data and compute to learn from scratch; using a pre-trained foundation model via API gets state-of-the-art language understanding for a few cents in tokens. It would not beat training your own model if I had a large labeled dataset for a narrow, well-defined task (e.g. classifying loan letters into 3 fixed risk categories), where a small trained classifier would be cheaper to run at scale, faster, fully under my control with no third-party data exposure, and not dependent on an external provider's availability or pricing changes.
